In [1]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import LayerIntegratedGradients

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / "models" / "distilbert" / "final_model"
DATA_DIR = PROJECT_ROOT / "dataset" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "xai"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 96

print("=" * 55)
print("XAI ENVIRONMENT")
print("=" * 55)
print("Device:", DEVICE)
print("Model directory:", MODEL_DIR)
print("Results directory:", RESULTS_DIR)
print("=" * 55)

c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


XAI ENVIRONMENT
Device: cuda
Model directory: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\models\distilbert\final_model
Results directory: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai


In [2]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR)
)

model.to(DEVICE)
model.eval()

print("=" * 55)
print("FINE-TUNED DISTILBERT LOADED")
print("=" * 55)
print("Model:", model.config.model_type)
print("Classes:", model.config.id2label)
print("Device:", next(model.parameters()).device)
print("=" * 55)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4522.02it/s]


FINE-TUNED DISTILBERT LOADED
Model: distilbert
Classes: {0: 'Benign', 1: 'Malicious'}
Device: cuda:0


In [3]:
test_df = pd.read_csv(DATA_DIR / "test.csv")

benign_df = test_df[test_df["label_binary"] == 0].sample(
    n=10, random_state=SEED
)

malicious_df = test_df[test_df["label_binary"] == 1].sample(
    n=10, random_state=SEED
)

xai_df = pd.concat(
    [benign_df, malicious_df],
    ignore_index=True
).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("=" * 55)
print("XAI SAMPLE SELECTED")
print("=" * 55)
print("Total URLs:", len(xai_df))
print("Benign:", (xai_df["label_binary"] == 0).sum())
print("Malicious:", (xai_df["label_binary"] == 1).sum())
print("=" * 55)

XAI SAMPLE SELECTED
Total URLs: 20
Benign: 10
Malicious: 10


In [4]:
@torch.no_grad()
def predict_url(url):
    encoded = tokenizer(
        str(url),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    encoded = {
        key: value.to(DEVICE)
        for key, value in encoded.items()
        if key in ("input_ids", "attention_mask")
    }

    outputs = model(**encoded)
    probabilities = torch.softmax(outputs.logits, dim=-1)[0]

    predicted_class = int(torch.argmax(probabilities).item())

    return {
        "prediction": predicted_class,
        "label": "Malicious" if predicted_class == 1 else "Benign",
        "confidence": float(probabilities[predicted_class].item()),
        "malicious_probability": float(probabilities[1].item())
    }


sample_url = str(xai_df.iloc[0]["clean_url"])
result = predict_url(sample_url)

print("=" * 55)
print("PREDICTION TEST")
print("=" * 55)
print("True label:", int(xai_df.iloc[0]["label_binary"]))
print("Prediction:", result["label"])
print("Confidence:", round(result["confidence"], 4))
print("=" * 55)

PREDICTION TEST
True label: 0
Prediction: Benign
Confidence: 0.9995


In [10]:
def forward_func(input_ids, attention_mask, target_class):
    outputs = model(
        input_ids=input_ids.long(),
        attention_mask=attention_mask.long()
    )

    return outputs.logits[:, target_class]


lig = LayerIntegratedGradients(
    forward_func,
    model.get_input_embeddings()
)


def explain_url_ig(url, n_steps=64):
    encoded = tokenizer(
        str(url),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits

        predicted_class = int(logits.argmax(dim=-1).item())

    baseline_ids = torch.full_like(
        input_ids,
        tokenizer.pad_token_id
    )

    baseline_ids[:, 0] = input_ids[:, 0]

    if input_ids[0, -1].item() == tokenizer.sep_token_id:
        baseline_ids[:, -1] = input_ids[:, -1]

    model.zero_grad(set_to_none=True)

    start = time.perf_counter()

    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask, predicted_class),
        n_steps=n_steps,
        internal_batch_size=1,
        return_convergence_delta=True
    )

    runtime = time.perf_counter() - start

    token_scores = attributions.sum(dim=-1).squeeze(0)

    tokens = tokenizer.convert_ids_to_tokens(
        input_ids.squeeze(0).tolist()
    )

    scores = token_scores.detach().cpu().numpy()

    special_ids = set(tokenizer.all_special_ids)

    token_results = []

    for token_id, token, score in zip(
        input_ids.squeeze(0).tolist(),
        tokens,
        scores
    ):
        if token_id in special_ids:
            continue

        token_results.append({
            "token": token,
            "attribution": float(score)
        })

    return {
        "url": str(url),
        "predicted_class": predicted_class,
        "tokens": token_results,
        "runtime_seconds": runtime,
        "convergence_delta": float(delta.detach().cpu().item())
    }

In [6]:
sample_url = str(xai_df.iloc[0]["clean_url"])

ig_result = explain_url_ig(
    sample_url,
    n_steps=16
)

token_df = pd.DataFrame(ig_result["tokens"])

token_df["absolute_attribution"] = (
    token_df["attribution"].abs()
)

top_tokens = token_df.sort_values(
    "absolute_attribution",
    ascending=False
).head(10)

print("=" * 65)
print("INTEGRATED GRADIENTS EXPLANATION")
print("=" * 65)

print("URL:", sample_url)
print("Predicted class:", ig_result["predicted_class"])
print("Runtime:", round(ig_result["runtime_seconds"], 3), "seconds")
print("Convergence delta:", round(ig_result["convergence_delta"], 6))

print("\nTOP 10 INFLUENTIAL TOKENS")
print(top_tokens[["token", "attribution"]].to_string(index=False))

print("=" * 65)

INTEGRATED GRADIENTS EXPLANATION
URL: platteriverdecoyco.com/demo.html
Predicted class: 0
Runtime: 0.439 seconds
Convergence delta: -2.597236

TOP 10 INFLUENTIAL TOKENS
token  attribution
    /     3.606741
    .    -0.461222
    .     0.356366
  com     0.159899
 html     0.123837
 ##de    -0.122453
 ##co    -0.114006
 demo    -0.112709
 ##co    -0.110868
##eri    -0.098674


In [11]:
ig_results = []

start_time = time.perf_counter()

for index, row in xai_df.iterrows():
    url = str(row["clean_url"])

    explanation = explain_url_ig(
        url,
        n_steps=64
    )

    ig_results.append({
        "sample_id": int(index),
        "url": url,
        "true_label": int(row["label_binary"]),
        "predicted_class": explanation["predicted_class"],
        "runtime_seconds": explanation["runtime_seconds"],
        "convergence_delta": explanation["convergence_delta"],
        "tokens": explanation["tokens"]
    })

total_time = time.perf_counter() - start_time

output_file = RESULTS_DIR / "integrated_gradients_results.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(ig_results, f, indent=2, ensure_ascii=False)

runtime_df = pd.DataFrame([
    {
        "sample_id": item["sample_id"],
        "runtime_seconds": item["runtime_seconds"],
        "absolute_delta": abs(item["convergence_delta"])
    }
    for item in ig_results
])

print("=" * 60)
print("INTEGRATED GRADIENTS PILOT COMPLETED")
print("=" * 60)
print("URLs explained:", len(ig_results))
print("Total time:", round(total_time, 2), "seconds")
print("Average explanation time:",
      round(runtime_df["runtime_seconds"].mean(), 3), "seconds")
print("Median explanation time:",
      round(runtime_df["runtime_seconds"].median(), 3), "seconds")
print("Mean absolute convergence delta:",
      round(runtime_df["absolute_delta"].mean(), 6))
print("Saved:", output_file)
print("=" * 60)

INTEGRATED GRADIENTS PILOT COMPLETED
URLs explained: 20
Total time: 13.88 seconds
Average explanation time: 0.686 seconds
Median explanation time: 0.665 seconds
Mean absolute convergence delta: 0.030697
Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\integrated_gradients_results.json


In [8]:
import pandas as pd

sample_url = str(xai_df.iloc[0]["clean_url"])

step_results = []

for steps in [16, 32, 64, 128]:

    result = explain_url_ig(
        sample_url,
        n_steps=steps
    )

    step_results.append({
        "n_steps": steps,
        "convergence_delta": result["convergence_delta"],
        "absolute_delta": abs(result["convergence_delta"]),
        "runtime_seconds": result["runtime_seconds"]
    })

convergence_df = pd.DataFrame(step_results)

print("=" * 65)
print("INTEGRATED GRADIENTS CONVERGENCE ANALYSIS")
print("=" * 65)

print(convergence_df.round(6).to_string(index=False))

print("=" * 65)

INTEGRATED GRADIENTS CONVERGENCE ANALYSIS
 n_steps  convergence_delta  absolute_delta  runtime_seconds
      16          -2.597236        2.597236         0.314037
      32           0.021574        0.021574         0.386268
      64           0.002699        0.002699         0.754956
     128          -0.008604        0.008604         1.359484


In [9]:
sample_url = str(xai_df.iloc[0]["clean_url"])

encoded = tokenizer(
    sample_url,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH
)

input_ids = encoded["input_ids"].to(DEVICE)
attention_mask = encoded["attention_mask"].to(DEVICE)

baseline_ids = torch.full_like(
    input_ids,
    tokenizer.pad_token_id
)

baseline_ids[:, 0] = input_ids[:, 0]

if input_ids[0, -1].item() == tokenizer.sep_token_id:
    baseline_ids[:, -1] = input_ids[:, -1]

with torch.no_grad():

    actual_logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    ).logits

    baseline_logits = model(
        input_ids=baseline_ids,
        attention_mask=attention_mask
    ).logits

    target_class = int(actual_logits.argmax(dim=-1).item())

    output_difference = (
        actual_logits[0, target_class]
        - baseline_logits[0, target_class]
    ).item()

ig_check = explain_url_ig(
    sample_url,
    n_steps=128
)

attribution_sum = sum(
    item["attribution"]
    for item in ig_check["tokens"]
)

print("=" * 65)
print("INTEGRATED GRADIENTS COMPLETENESS CHECK")
print("=" * 65)

print("Target class:", target_class)
print("Output difference:", round(output_difference, 6))
print("Attribution sum (excluding special tokens):",
      round(attribution_sum, 6))
print("Captum convergence delta:",
      round(ig_check["convergence_delta"], 6))

print("=" * 65)

INTEGRATED GRADIENTS COMPLETENESS CHECK
Target class: 0
Output difference: 5.697853
Attribution sum (excluding special tokens): 5.68925
Captum convergence delta: -0.008604


In [12]:
validation_results = []

for index, row in xai_df.iterrows():

    result = explain_url_ig(
        str(row["clean_url"]),
        n_steps=64
    )

    validation_results.append({
        "sample_id": index,
        "absolute_delta": abs(result["convergence_delta"]),
        "runtime_seconds": result["runtime_seconds"]
    })

validation_df = pd.DataFrame(validation_results)

print("=" * 60)
print("IG VALIDATION — 64 STEPS")
print("=" * 60)
print("URLs tested:", len(validation_df))
print("Mean absolute delta:", validation_df["absolute_delta"].mean())
print("Median absolute delta:", validation_df["absolute_delta"].median())
print("Maximum absolute delta:", validation_df["absolute_delta"].max())
print("Mean runtime:", validation_df["runtime_seconds"].mean())
print("=" * 60)

IG VALIDATION — 64 STEPS
URLs tested: 20
Mean absolute delta: 0.030697421729564668
Median absolute delta: 0.017979979515075684
Maximum absolute delta: 0.12447413802146912
Mean runtime: 0.6913837100029923


Import LIME

In [13]:
from lime.lime_text import LimeTextExplainer
import re

print("LIME imported successfully")

LIME imported successfully


In [14]:
@torch.no_grad()
def predict_proba_lime(texts, batch_size=16):

    probabilities = []

    for start in range(0, len(texts), batch_size):

        batch = [str(x) for x in texts[start:start + batch_size]]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
            if key in ("input_ids", "attention_mask")
        }

        outputs = model(**encoded)

        probs = torch.softmax(
            outputs.logits,
            dim=-1
        ).cpu().numpy()

        probabilities.append(probs)

    return np.vstack(probabilities)


test_probs = predict_proba_lime([
    "https://www.google.com",
    "http://example.com/login"
])

print("Prediction shape:", test_probs.shape)
print("Class probabilities:")
print(test_probs)

Prediction shape: (2, 2)
Class probabilities:
[[3.863692e-04 9.996137e-01]
 [3.741371e-03 9.962586e-01]]


In [15]:
lime_explainer = LimeTextExplainer(
    class_names=["Benign", "Malicious"],
    split_expression=r"(?<=[/:.?=&_\-])|(?=[/:.?=&_\-])",
    bow=False,
    random_state=SEED
)

print("LIME explainer initialized")

LIME explainer initialized


In [16]:
def explain_url_lime(url, num_samples=300):

    url = str(url)

    prediction = predict_proba_lime([url])[0]
    predicted_class = int(np.argmax(prediction))

    start_time = time.perf_counter()

    explanation = lime_explainer.explain_instance(
        url,
        classifier_fn=predict_proba_lime,
        labels=(predicted_class,),
        num_features=20,
        num_samples=num_samples
    )

    runtime = time.perf_counter() - start_time

    token_scores = explanation.as_list(
        label=predicted_class
    )

    return {
        "url": url,
        "predicted_class": predicted_class,
        "runtime_seconds": runtime,
        "local_fidelity_r2": float(explanation.score),
        "tokens": [
            {
                "token": token,
                "attribution": float(score)
            }
            for token, score in token_scores
        ]
    }


sample_url = str(xai_df.iloc[0]["clean_url"])

lime_result = explain_url_lime(
    sample_url,
    num_samples=300
)

print("=" * 65)
print("LIME EXPLANATION")
print("=" * 65)

print("URL:", sample_url)
print("Predicted class:", lime_result["predicted_class"])
print("Runtime:", round(lime_result["runtime_seconds"], 3), "seconds")
print("Local surrogate R²:", round(lime_result["local_fidelity_r2"], 4))

print("\nTOP INFLUENTIAL FEATURES")

lime_token_df = pd.DataFrame(lime_result["tokens"])
print(lime_token_df.to_string(index=False))

print("=" * 65)

LIME EXPLANATION
URL: platteriverdecoyco.com/demo.html
Predicted class: 0
Runtime: 0.859 seconds
Local surrogate R²: 0.5559

TOP INFLUENTIAL FEATURES
             token  attribution
               com     0.001473
platteriverdecoyco     0.001128
              html     0.000336
              demo    -0.000045


In [17]:
lime_results = []

start_time = time.perf_counter()

for index, row in xai_df.iterrows():

    explanation = explain_url_lime(
        str(row["clean_url"]),
        num_samples=300
    )

    lime_results.append({
        "sample_id": int(index),
        "url": str(row["clean_url"]),
        "true_label": int(row["label_binary"]),
        **explanation
    })

total_time = time.perf_counter() - start_time

output_file = RESULTS_DIR / "lime_results.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(lime_results, f, indent=2, ensure_ascii=False)

lime_runtime_df = pd.DataFrame([
    {
        "runtime_seconds": item["runtime_seconds"],
        "local_fidelity_r2": item["local_fidelity_r2"]
    }
    for item in lime_results
])

print("=" * 60)
print("LIME PILOT COMPLETED")
print("=" * 60)
print("URLs explained:", len(lime_results))
print("Total runtime:", round(total_time, 2), "seconds")
print("Average runtime:", round(lime_runtime_df["runtime_seconds"].mean(), 3))
print("Median runtime:", round(lime_runtime_df["runtime_seconds"].median(), 3))
print("Mean local surrogate R²:",
      round(lime_runtime_df["local_fidelity_r2"].mean(), 4))
print("Saved:", output_file)
print("=" * 60)

LIME PILOT COMPLETED
URLs explained: 20
Total runtime: 21.39 seconds
Average runtime: 1.062
Median runtime: 0.866
Mean local surrogate R²: 0.6772
Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\lime_results.json


In [18]:
import shap

shap_masker = shap.maskers.Text(tokenizer)

shap_explainer = shap.Explainer(
    predict_proba_lime,
    shap_masker,
    output_names=["Benign", "Malicious"],
    algorithm="partition"
)

print("=" * 55)
print("SHAP EXPLAINER INITIALIZED")
print("=" * 55)
print("Method: Partition SHAP")
print("Masker: DistilBERT tokenizer")
print("Device:", DEVICE)
print("=" * 55)

SHAP EXPLAINER INITIALIZED
Method: Partition SHAP
Masker: DistilBERT tokenizer
Device: cuda


In [19]:
def explain_url_shap(url, max_evals=300):

    url = str(url)

    prediction = predict_proba_lime([url])[0]
    predicted_class = int(np.argmax(prediction))

    start_time = time.perf_counter()

    shap_values = shap_explainer(
        [url],
        max_evals=max_evals,
        batch_size=16
    )

    runtime = time.perf_counter() - start_time

    token_values = np.asarray(
        shap_values.values[0, :, predicted_class]
    )

    tokens = shap_values.data[0]

    token_results = []

    for token, value in zip(tokens, token_values):
        token_results.append({
            "token": str(token),
            "attribution": float(value)
        })

    return {
        "url": url,
        "predicted_class": predicted_class,
        "runtime_seconds": runtime,
        "tokens": token_results
    }


sample_url = str(xai_df.iloc[0]["clean_url"])

shap_result = explain_url_shap(
    sample_url,
    max_evals=300
)

shap_token_df = pd.DataFrame(shap_result["tokens"])

shap_token_df["absolute_attribution"] = (
    shap_token_df["attribution"].abs()
)

top_shap_tokens = shap_token_df.sort_values(
    "absolute_attribution",
    ascending=False
).head(10)

print("=" * 65)
print("SHAP EXPLANATION")
print("=" * 65)

print("URL:", sample_url)
print("Predicted class:", shap_result["predicted_class"])
print("Runtime:", round(shap_result["runtime_seconds"], 3), "seconds")

print("\nTOP INFLUENTIAL TOKENS")
print(
    top_shap_tokens[
        ["token", "attribution"]
    ].to_string(index=False)
)

print("=" * 65)

SHAP EXPLANATION
URL: platteriverdecoyco.com/demo.html
Predicted class: 0
Runtime: 7.696 seconds

TOP INFLUENTIAL TOKENS
token  attribution
    /     0.888551
  ver    -0.029597
  eri    -0.019675
 demo    -0.015070
  com     0.013346
   co     0.009658
    .    -0.009041
 html    -0.007907
    y     0.007064
    .    -0.004051


In [20]:
shap_results = []

start_time = time.perf_counter()

for index, row in xai_df.iterrows():

    explanation = explain_url_shap(
        str(row["clean_url"]),
        max_evals=300
    )

    shap_results.append({
        "sample_id": int(index),
        "url": str(row["clean_url"]),
        "true_label": int(row["label_binary"]),
        **explanation
    })

total_time = time.perf_counter() - start_time

output_file = RESULTS_DIR / "shap_results.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        shap_results,
        f,
        indent=2,
        ensure_ascii=False
    )

shap_runtime_df = pd.DataFrame([
    {
        "runtime_seconds": item["runtime_seconds"]
    }
    for item in shap_results
])

print("=" * 60)
print("SHAP PILOT COMPLETED")
print("=" * 60)

print("URLs explained:", len(shap_results))
print("Total runtime:", round(total_time, 2), "seconds")
print(
    "Average runtime:",
    round(shap_runtime_df["runtime_seconds"].mean(), 3)
)
print(
    "Median runtime:",
    round(shap_runtime_df["runtime_seconds"].median(), 3)
)

print("Saved:", output_file)
print("=" * 60)

SHAP PILOT COMPLETED
URLs explained: 20
Total runtime: 13.63 seconds
Average runtime: 0.674
Median runtime: 0.592
Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\shap_results.json


In [21]:
import json
import numpy as np
import pandas as pd

methods = {
    "Integrated Gradients": RESULTS_DIR / "integrated_gradients_results.json",
    "LIME": RESULTS_DIR / "lime_results.json",
    "SHAP": RESULTS_DIR / "shap_results.json"
}

summary_rows = []

for method_name, file_path in methods.items():

    with open(file_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    runtimes = np.array([
        record["runtime_seconds"]
        for record in records
    ], dtype=float)

    summary_rows.append({
        "Method": method_name,
        "Samples": len(records),
        "Mean Runtime (s)": float(np.mean(runtimes)),
        "Median Runtime (s)": float(np.median(runtimes)),
        "Std Runtime (s)": float(np.std(runtimes, ddof=1)),
        "Min Runtime (s)": float(np.min(runtimes)),
        "Max Runtime (s)": float(np.max(runtimes))
    })

xai_comparison_df = pd.DataFrame(summary_rows)

output_path = RESULTS_DIR / "xai_runtime_comparison.csv"

xai_comparison_df.to_csv(output_path, index=False)

print("=" * 95)
print("XAI COMPUTATIONAL COST COMPARISON")
print("=" * 95)
print(xai_comparison_df.round(4).to_string(index=False))
print("=" * 95)
print("Saved:", output_path)

XAI COMPUTATIONAL COST COMPARISON
              Method  Samples  Mean Runtime (s)  Median Runtime (s)  Std Runtime (s)  Min Runtime (s)  Max Runtime (s)
Integrated Gradients       20            0.6860              0.6646           0.0872           0.6189           0.9480
                LIME       20            1.0620              0.8659           0.5475           0.3174           2.0155
                SHAP       20            0.6742              0.5922           0.3989           0.0727           1.7992
Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\xai_runtime_comparison.csv


In [22]:
import json
import numpy as np
import pandas as pd

xai_files = {
    "IG": RESULTS_DIR / "integrated_gradients_results.json",
    "LIME": RESULTS_DIR / "lime_results.json",
    "SHAP": RESULTS_DIR / "shap_results.json"
}

all_explanations = {}

for method, path in xai_files.items():
    with open(path, "r", encoding="utf-8") as f:
        all_explanations[method] = json.load(f)

    print(method, "loaded:", len(all_explanations[method]))

assert all(len(v) == 20 for v in all_explanations.values())

for i in range(20):
    urls = {
        all_explanations[method][i]["url"]
        for method in all_explanations
    }
    assert len(urls) == 1, f"URL mismatch at sample {i}"

print("All explanations correspond to the same 20 URLs.")

IG loaded: 20
LIME loaded: 20
SHAP loaded: 20
All explanations correspond to the same 20 URLs.


In [23]:
import re

def mask_important_features(url, token_scores, fraction=0.2):
    url = str(url)

    positive_features = [
        item for item in token_scores
        if item["attribution"] > 0 and item["token"].strip()
    ]

    positive_features.sort(
        key=lambda x: x["attribution"],
        reverse=True
    )

    n_mask = max(1, int(np.ceil(len(positive_features) * fraction)))

    masked_url = url

    for item in positive_features[:n_mask]:
        token = item["token"].replace("##", "").strip()

        if not token:
            continue

        position = masked_url.find(token)

        if position >= 0:
            masked_url = (
                masked_url[:position]
                + ("x" * len(token))
                + masked_url[position + len(token):]
            )

    return masked_url

In [24]:
faithfulness_rows = []

for method, explanations in all_explanations.items():

    for item in explanations:

        original_url = item["url"]

        masked_url = mask_important_features(
            original_url,
            item["tokens"],
            fraction=0.2
        )

        original_probs = predict_proba_lime([original_url])[0]
        masked_probs = predict_proba_lime([masked_url])[0]

        target_class = int(np.argmax(original_probs))

        confidence_drop = (
            original_probs[target_class]
            - masked_probs[target_class]
        )

        faithfulness_rows.append({
            "Method": method,
            "URL": original_url,
            "Original Confidence": float(original_probs[target_class]),
            "Masked Confidence": float(masked_probs[target_class]),
            "Confidence Drop": float(confidence_drop),
            "Changed": original_url != masked_url
        })

faithfulness_df = pd.DataFrame(faithfulness_rows)

print("=" * 75)
print("EXPLORATORY XAI DELETION TEST")
print("=" * 75)

summary = faithfulness_df.groupby("Method").agg(
    Samples=("URL", "count"),
    Mean_Confidence_Drop=("Confidence Drop", "mean"),
    Median_Confidence_Drop=("Confidence Drop", "median"),
    Changed_URLs=("Changed", "sum")
)

print(summary.round(6).to_string())

output_path = RESULTS_DIR / "xai_faithfulness_pilot.csv"
faithfulness_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

EXPLORATORY XAI DELETION TEST
        Samples  Mean_Confidence_Drop  Median_Confidence_Drop  Changed_URLs
Method                                                                     
IG           20              0.854794                0.996533            20
LIME         20              0.193308                0.000431            20
SHAP         20              0.792707                0.997052            20

Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\xai_faithfulness_pilot.csv


In [25]:
import numpy as np
import pandas as pd

diagnostic_rows = []

for method, explanations in all_explanations.items():

    for item in explanations:

        original_url = item["url"]

        masked_url = mask_important_features(
            original_url,
            item["tokens"],
            fraction=0.2
        )

        original_probs = predict_proba_lime([original_url])[0]
        masked_probs = predict_proba_lime([masked_url])[0]

        original_class = int(np.argmax(original_probs))
        masked_class = int(np.argmax(masked_probs))

        changed_characters = sum(
            a != b
            for a, b in zip(original_url, masked_url)
        )

        changed_fraction = (
            changed_characters / len(original_url)
            if original_url else 0
        )

        diagnostic_rows.append({
            "Method": method,
            "URL": original_url,
            "Original Prediction": original_class,
            "Masked Prediction": masked_class,
            "Prediction Flip": int(original_class != masked_class),
            "Changed Characters": changed_characters,
            "Changed Fraction": changed_fraction,
            "Confidence Drop": float(
                original_probs[original_class]
                - masked_probs[original_class]
            )
        })

diagnostic_df = pd.DataFrame(diagnostic_rows)

summary = diagnostic_df.groupby("Method").agg(
    Samples=("URL", "count"),
    Mean_Changed_Characters=("Changed Characters", "mean"),
    Mean_Changed_Fraction=("Changed Fraction", "mean"),
    Prediction_Flip_Rate=("Prediction Flip", "mean"),
    Mean_Confidence_Drop=("Confidence Drop", "mean")
)

print("=" * 90)
print("XAI MASKING DIAGNOSTIC")
print("=" * 90)

print(summary.round(6).to_string())

print("=" * 90)

output_path = RESULTS_DIR / "xai_masking_diagnostic.csv"

diagnostic_df.to_csv(output_path, index=False)

print("Saved:", output_path)

XAI MASKING DIAGNOSTIC
        Samples  Mean_Changed_Characters  Mean_Changed_Fraction  Prediction_Flip_Rate  Mean_Confidence_Drop
Method                                                                                                     
IG           20                     7.00               0.113054                  0.85              0.854794
LIME         20                     7.55               0.130816                  0.20              0.193308
SHAP         20                     7.50               0.123066                  0.80              0.792707
Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\xai_masking_diagnostic.csv


In [26]:
from urllib.parse import urlsplit

def extract_url_components(url):
    url = str(url)

    # urlsplit needs a scheme to recognize the hostname.
    normalized = url if "://" in url else "http://" + url
    parsed = urlsplit(normalized)

    components = {
        "domain": parsed.netloc,
        "path": parsed.path,
        "query": parsed.query,
        "fragment": parsed.fragment
    }

    return components


sample_url = str(xai_df.iloc[0]["clean_url"])

components = extract_url_components(sample_url)

print("=" * 60)
print("URL COMPONENT EXTRACTION")
print("=" * 60)
print("URL:", sample_url)

for name, value in components.items():
    print(f"{name}: {value}")

print("=" * 60)

URL COMPONENT EXTRACTION
URL: platteriverdecoyco.com/demo.html
domain: platteriverdecoyco.com
path: /demo.html
query: 
fragment: 


In [27]:
def mask_url_component(url, component_name):
    url = str(url)

    normalized = url if "://" in url else "http://" + url
    parsed = urlsplit(normalized)

    component_values = {
        "domain": parsed.netloc,
        "path": parsed.path,
        "query": parsed.query,
        "fragment": parsed.fragment
    }

    component = component_values.get(component_name, "")

    if not component:
        return url

    if component_name == "domain":
        start = url.find(component)
    elif component_name == "path":
        start = url.find(component, url.find(parsed.netloc) + len(parsed.netloc))
    elif component_name == "query":
        start = url.find("?" + component)
        if start >= 0:
            start += 1
    else:
        start = url.find("#" + component)
        if start >= 0:
            start += 1

    if start < 0:
        return url

    return (
        url[:start]
        + ("x" * len(component))
        + url[start + len(component):]
    )

In [28]:
component_rows = []

component_names = ["domain", "path", "query", "fragment"]

for index, row in xai_df.iterrows():

    url = str(row["clean_url"])

    original_probs = predict_proba_lime([url])[0]
    target_class = int(np.argmax(original_probs))

    for component in component_names:

        masked_url = mask_url_component(url, component)

        if masked_url == url:
            continue

        masked_probs = predict_proba_lime([masked_url])[0]

        confidence_drop = (
            original_probs[target_class]
            - masked_probs[target_class]
        )

        component_rows.append({
            "Sample ID": int(index),
            "Component": component,
            "Original URL": url,
            "Masked URL": masked_url,
            "Confidence Drop": float(confidence_drop),
            "Prediction Flip": int(
                np.argmax(masked_probs) != target_class
            )
        })

component_df = pd.DataFrame(component_rows)

summary = component_df.groupby("Component").agg(
    Samples=("Sample ID", "count"),
    Mean_Confidence_Drop=("Confidence Drop", "mean"),
    Prediction_Flip_Rate=("Prediction Flip", "mean")
)

print("=" * 80)
print("URL COMPONENT SENSITIVITY ANALYSIS")
print("=" * 80)

print(summary.round(6).to_string())

output_path = RESULTS_DIR / "url_component_sensitivity.csv"

component_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

URL COMPONENT SENSITIVITY ANALYSIS
           Samples  Mean_Confidence_Drop  Prediction_Flip_Rate
Component                                                     
domain          20              0.225304                  0.25
path            20              0.460332                  0.45
query            4              0.000344                  0.00

Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\url_component_sensitivity.csv


In [29]:
component_reference = component_df.copy()

component_reference["Absolute Effect"] = (
    component_reference["Confidence Drop"].abs()
)

component_reference["Component Rank"] = (
    component_reference.groupby("Sample ID")["Absolute Effect"]
    .rank(method="dense", ascending=False)
)

reference_path = RESULTS_DIR / "component_occlusion_reference.csv"

component_reference.to_csv(reference_path, index=False)

print("=" * 75)
print("COMPONENT OCCLUSION REFERENCE")
print("=" * 75)

summary = component_reference.groupby("Component").agg(
    Samples=("Sample ID", "count"),
    Mean_Absolute_Effect=("Absolute Effect", "mean"),
    Median_Absolute_Effect=("Absolute Effect", "median"),
    Mean_Rank=("Component Rank", "mean")
)

print(summary.round(6).to_string())

print("\nSaved:", reference_path)

COMPONENT OCCLUSION REFERENCE
           Samples  Mean_Absolute_Effect  Median_Absolute_Effect  Mean_Rank
Component                                                                  
domain          20              0.225308                0.000446       1.65
path            20              0.462224                0.131338       1.60
query            4              0.000344                0.000232       1.75

Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\component_occlusion_reference.csv


In [30]:
top_components = (
    component_reference
    .sort_values(
        ["Sample ID", "Absolute Effect"],
        ascending=[True, False]
    )
    .groupby("Sample ID")
    .first()
    .reset_index()
)

print("=" * 75)
print("MOST SENSITIVE COMPONENT PER URL")
print("=" * 75)

print(
    top_components[
        ["Sample ID", "Component", "Confidence Drop", "Prediction Flip"]
    ].to_string(index=False)
)

print("=" * 75)

MOST SENSITIVE COMPONENT PER URL
 Sample ID Component  Confidence Drop  Prediction Flip
         0      path         0.999027                1
         1    domain         0.000355                0
         2    domain         0.958788                1
         3      path         0.998940                1
         4      path         0.998614                1
         5      path         0.250495                0
         6    domain         0.000449                0
         7      path         0.998357                1
         8    domain         0.027202                0
         9    domain         0.992580                1
        10    domain         0.991409                1
        11      path         0.998946                1
        12      path         0.998585                1
        13    domain         0.000102                0
        14      path         0.998991                1
        15     query         0.000259                0
        16      path         0.9

In [31]:
from urllib.parse import urlsplit

def get_component_spans(url):
    url = str(url)

    normalized = url if "://" in url else "http://" + url
    parsed = urlsplit(normalized)

    spans = {}

    scheme_offset = len(url) - len(normalized)

    if parsed.netloc:
        start = normalized.find(parsed.netloc)
        spans["domain"] = (
            start + scheme_offset,
            start + scheme_offset + len(parsed.netloc)
        )

    search_start = normalized.find(parsed.netloc) + len(parsed.netloc)

    if parsed.path:
        start = normalized.find(parsed.path, search_start)
        if start >= 0:
            spans["path"] = (
                start + scheme_offset,
                start + scheme_offset + len(parsed.path)
            )

    if parsed.query:
        start = normalized.find("?" + parsed.query, search_start)
        if start >= 0:
            spans["query"] = (
                start + 1 + scheme_offset,
                start + 1 + scheme_offset + len(parsed.query)
            )

    if parsed.fragment:
        start = normalized.find("#" + parsed.fragment, search_start)
        if start >= 0:
            spans["fragment"] = (
                start + 1 + scheme_offset,
                start + 1 + scheme_offset + len(parsed.fragment)
            )

    return spans


sample_url = str(xai_df.iloc[0]["clean_url"])

print("URL:", sample_url)
print("Component spans:", get_component_spans(sample_url))

URL: platteriverdecoyco.com/demo.html
Component spans: {'domain': (0, 22), 'path': (22, 32)}


In [32]:
def explain_ig_components(url, n_steps=64):
    url = str(url)

    encoded = tokenizer(
        url,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True
    )

    offsets = encoded.pop("offset_mapping")[0].tolist()

    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits

        predicted_class = int(logits.argmax(dim=-1).item())

    baseline_ids = torch.full_like(
        input_ids,
        tokenizer.pad_token_id
    )

    baseline_ids[:, 0] = input_ids[:, 0]

    if input_ids[0, -1].item() == tokenizer.sep_token_id:
        baseline_ids[:, -1] = input_ids[:, -1]

    model.zero_grad(set_to_none=True)

    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask, predicted_class),
        n_steps=n_steps,
        internal_batch_size=1,
        return_convergence_delta=True
    )

    token_scores = (
        attributions.sum(dim=-1)[0]
        .detach().cpu().numpy()
    )

    spans = get_component_spans(url)

    component_scores = {
        name: 0.0
        for name in spans
    }

    for (start, end), score in zip(offsets, token_scores):

        if start == end:
            continue

        token_length = end - start

        for name, (comp_start, comp_end) in spans.items():

            overlap = max(
                0,
                min(end, comp_end) - max(start, comp_start)
            )

            if overlap > 0:
                component_scores[name] += (
                    float(score) * overlap / token_length
                )

    return {
        "url": url,
        "predicted_class": predicted_class,
        "component_scores": component_scores,
        "convergence_delta": float(delta.detach().cpu().item())
    }


result = explain_ig_components(sample_url)

print("=" * 65)
print("IG COMPONENT-LEVEL EXPLANATION")
print("=" * 65)
print("URL:", sample_url)
print("Predicted class:", result["predicted_class"])
print("Component scores:", result["component_scores"])
print("Convergence delta:", result["convergence_delta"])

IG COMPONENT-LEVEL EXPLANATION
URL: platteriverdecoyco.com/demo.html
Predicted class: 0
Component scores: {'domain': 0.4984471770003438, 'path': 5.202105022966862}
Convergence delta: 0.0026993751525878906


In [33]:
ig_component_results = []

for index, row in xai_df.iterrows():

    explanation = explain_ig_components(
        str(row["clean_url"]),
        n_steps=64
    )

    ig_component_results.append({
        "sample_id": int(index),
        "true_label": int(row["label_binary"]),
        **explanation
    })

output_path = RESULTS_DIR / "ig_component_attributions.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        ig_component_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 65)
print("IG COMPONENT ATTRIBUTIONS COMPLETED")
print("=" * 65)
print("URLs explained:", len(ig_component_results))
print("Saved:", output_path)

IG COMPONENT ATTRIBUTIONS COMPLETED
URLs explained: 20
Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\ig_component_attributions.json


In [34]:
def mask_components(url, components_to_mask):
    url = str(url)
    spans = get_component_spans(url)

    chars = list(url)

    for component in components_to_mask:
        if component not in spans:
            continue

        start, end = spans[component]

        for position in range(start, end):
            chars[position] = "x"

    return "".join(chars)


sample_url = str(xai_df.iloc[0]["clean_url"])

print("Original:", sample_url)
print("Domain masked:", mask_components(sample_url, ["domain"]))
print("Path masked:", mask_components(sample_url, ["path"]))
print("Both masked:", mask_components(sample_url, ["domain", "path"]))

Original: platteriverdecoyco.com/demo.html
Domain masked: xxxxxxxxxxxxxxxxxxxxxx/demo.html
Path masked: platteriverdecoyco.comxxxxxxxxxx
Both masked: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx


In [35]:
from sklearn.linear_model import Ridge

def explain_lime_components(url, num_samples=256, seed=42):
    url = str(url)
    components = list(get_component_spans(url).keys())
    n = len(components)

    if n == 0:
        return {}

    rng = np.random.default_rng(seed)

    masks = rng.integers(0, 2, size=(num_samples, n))
    masks[0] = 1
    masks[1] = 0

    perturbed_urls = []

    for mask in masks:
        removed = [
            components[i]
            for i in range(n)
            if mask[i] == 0
        ]
        perturbed_urls.append(mask_components(url, removed))

    probabilities = predict_proba_lime(perturbed_urls)

    target_class = int(
        np.argmax(predict_proba_lime([url])[0])
    )

    y = probabilities[:, target_class]

    distances = np.sqrt(
        np.mean((1 - masks) ** 2, axis=1)
    )

    weights = np.exp(-(distances ** 2) / (0.75 ** 2))

    surrogate = Ridge(alpha=0.01)
    surrogate.fit(masks, y, sample_weight=weights)

    return {
        "url": url,
        "predicted_class": target_class,
        "component_scores": {
            name: float(score)
            for name, score in zip(
                components,
                surrogate.coef_
            )
        },
        "local_r2": float(
            surrogate.score(masks, y, sample_weight=weights)
        )
    }


lime_component_example = explain_lime_components(sample_url)

print("LIME-style component scores:")
print(lime_component_example)

LIME-style component scores:
{'url': 'platteriverdecoyco.com/demo.html', 'predicted_class': 0, 'component_scores': {'domain': 0.6667711385332785, 'path': 0.7315962398489014}, 'local_r2': 0.8356897258339552}


In [36]:
from itertools import combinations
from math import factorial

def explain_shap_components(url):
    url = str(url)

    components = list(get_component_spans(url).keys())
    n = len(components)

    if n == 0:
        return {}

    coalitions = []

    for size in range(n + 1):
        for subset in combinations(range(n), size):
            coalitions.append(frozenset(subset))

    perturbed_urls = []

    for coalition in coalitions:
        removed = [
            components[i]
            for i in range(n)
            if i not in coalition
        ]
        perturbed_urls.append(mask_components(url, removed))

    probabilities = predict_proba_lime(perturbed_urls)

    target_class = int(
        np.argmax(predict_proba_lime([url])[0])
    )

    values = {
        coalition: float(probabilities[j, target_class])
        for j, coalition in enumerate(coalitions)
    }

    scores = {}

    for i, component in enumerate(components):
        shapley_value = 0.0

        for coalition in coalitions:
            if i in coalition:
                continue

            size = len(coalition)

            weight = (
                factorial(size)
                * factorial(n - size - 1)
                / factorial(n)
            )

            with_component = frozenset(set(coalition) | {i})

            shapley_value += weight * (
                values[with_component] - values[coalition]
            )

        scores[component] = float(shapley_value)

    return {
        "url": url,
        "predicted_class": target_class,
        "component_scores": scores
    }


shap_component_example = explain_shap_components(sample_url)

print("Exact component Shapley scores:")
print(shap_component_example)

Exact component Shapley scores:
{'url': 'platteriverdecoyco.com/demo.html', 'predicted_class': 0, 'component_scores': {'domain': 0.4637991873169085, 'path': 0.5334892637474695}}


In [37]:
lime_component_results = []
shap_component_results = []

for index, row in xai_df.iterrows():
    url = str(row["clean_url"])

    lime_result = explain_lime_components(url, seed=SEED + index)
    shap_result = explain_shap_components(url)

    lime_component_results.append({
        "sample_id": int(index),
        "true_label": int(row["label_binary"]),
        **lime_result
    })

    shap_component_results.append({
        "sample_id": int(index),
        "true_label": int(row["label_binary"]),
        **shap_result
    })

for name, records in [
    ("lime_component_attributions.json", lime_component_results),
    ("shap_component_attributions.json", shap_component_results)
]:
    path = RESULTS_DIR / name

    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    print("Saved:", path)

print("Component-level LIME-style and SHAP explanations completed.")

Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\lime_component_attributions.json
Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\shap_component_attributions.json
Component-level LIME-style and SHAP explanations completed.


In [38]:
def mask_components(url, components_to_mask):
    url = str(url)
    spans = get_component_spans(url)

    chars = list(url)

    for component in components_to_mask:
        if component not in spans:
            continue

        start, end = spans[component]

        for position in range(start, end):
            chars[position] = "x"

    return "".join(chars)


sample_url = str(xai_df.iloc[0]["clean_url"])

print("Original:", sample_url)
print("Domain masked:", mask_components(sample_url, ["domain"]))
print("Path masked:", mask_components(sample_url, ["path"]))
print("Both masked:", mask_components(sample_url, ["domain", "path"]))

Original: platteriverdecoyco.com/demo.html
Domain masked: xxxxxxxxxxxxxxxxxxxxxx/demo.html
Path masked: platteriverdecoyco.comxxxxxxxxxx
Both masked: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx


In [39]:
from sklearn.linear_model import Ridge

def explain_lime_components(url, num_samples=256, seed=42):
    url = str(url)
    components = list(get_component_spans(url).keys())
    n = len(components)

    if n == 0:
        return {}

    rng = np.random.default_rng(seed)

    masks = rng.integers(0, 2, size=(num_samples, n))
    masks[0] = 1
    masks[1] = 0

    perturbed_urls = []

    for mask in masks:
        removed = [
            components[i]
            for i in range(n)
            if mask[i] == 0
        ]
        perturbed_urls.append(mask_components(url, removed))

    probabilities = predict_proba_lime(perturbed_urls)

    target_class = int(
        np.argmax(predict_proba_lime([url])[0])
    )

    y = probabilities[:, target_class]

    distances = np.sqrt(
        np.mean((1 - masks) ** 2, axis=1)
    )

    weights = np.exp(-(distances ** 2) / (0.75 ** 2))

    surrogate = Ridge(alpha=0.01)
    surrogate.fit(masks, y, sample_weight=weights)

    return {
        "url": url,
        "predicted_class": target_class,
        "component_scores": {
            name: float(score)
            for name, score in zip(
                components,
                surrogate.coef_
            )
        },
        "local_r2": float(
            surrogate.score(masks, y, sample_weight=weights)
        )
    }


lime_component_example = explain_lime_components(sample_url)

print("LIME-style component scores:")
print(lime_component_example)

LIME-style component scores:
{'url': 'platteriverdecoyco.com/demo.html', 'predicted_class': 0, 'component_scores': {'domain': 0.6667711385332785, 'path': 0.7315962398489014}, 'local_r2': 0.8356897258339552}


In [40]:
from itertools import combinations
from math import factorial

def explain_shap_components(url):
    url = str(url)

    components = list(get_component_spans(url).keys())
    n = len(components)

    if n == 0:
        return {}

    coalitions = []

    for size in range(n + 1):
        for subset in combinations(range(n), size):
            coalitions.append(frozenset(subset))

    perturbed_urls = []

    for coalition in coalitions:
        removed = [
            components[i]
            for i in range(n)
            if i not in coalition
        ]
        perturbed_urls.append(mask_components(url, removed))

    probabilities = predict_proba_lime(perturbed_urls)

    target_class = int(
        np.argmax(predict_proba_lime([url])[0])
    )

    values = {
        coalition: float(probabilities[j, target_class])
        for j, coalition in enumerate(coalitions)
    }

    scores = {}

    for i, component in enumerate(components):
        shapley_value = 0.0

        for coalition in coalitions:
            if i in coalition:
                continue

            size = len(coalition)

            weight = (
                factorial(size)
                * factorial(n - size - 1)
                / factorial(n)
            )

            with_component = frozenset(set(coalition) | {i})

            shapley_value += weight * (
                values[with_component] - values[coalition]
            )

        scores[component] = float(shapley_value)

    return {
        "url": url,
        "predicted_class": target_class,
        "component_scores": scores
    }


shap_component_example = explain_shap_components(sample_url)

print("Exact component Shapley scores:")
print(shap_component_example)

Exact component Shapley scores:
{'url': 'platteriverdecoyco.com/demo.html', 'predicted_class': 0, 'component_scores': {'domain': 0.4637991873169085, 'path': 0.5334892637474695}}


In [42]:
lime_component_results = []
shap_component_results = []

for index, row in xai_df.iterrows():
    url = str(row["clean_url"])

    lime_result = explain_lime_components(url, seed=SEED + index)
    shap_result = explain_shap_components(url)

    lime_component_results.append({
        "sample_id": int(index),
        "true_label": int(row["label_binary"]),
        **lime_result
    })

    shap_component_results.append({
        "sample_id": int(index),
        "true_label": int(row["label_binary"]),
        **shap_result
    })

for name, records in [
    ("lime_component_attributions.json", lime_component_results),
    ("shap_component_attributions.json", shap_component_results)
]:
    path = RESULTS_DIR / name

    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    print("Saved:", path)

print("Component-level LIME-style and SHAP explanations completed.")

Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\lime_component_attributions.json
Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\shap_component_attributions.json
Component-level LIME-style and SHAP explanations completed.


In [43]:
component_files = {
    "IG": "ig_component_attributions.json",
    "LIME": "lime_component_attributions.json",
    "SHAP": "shap_component_attributions.json"
}

component_explanations = {}

for method, filename in component_files.items():

    with open(RESULTS_DIR / filename, "r", encoding="utf-8") as f:
        component_explanations[method] = json.load(f)

    print(method, "loaded:", len(component_explanations[method]))

reference_df = pd.read_csv(
    RESULTS_DIR / "component_occlusion_reference.csv"
)

print("Reference rows:", len(reference_df))

IG loaded: 20
LIME loaded: 20
SHAP loaded: 20
Reference rows: 44


In [44]:
ranking_rows = []

for method, explanations in component_explanations.items():

    for item in explanations:

        sample_id = int(item["sample_id"])
        scores = item["component_scores"]

        ref_rows = reference_df[
            reference_df["Sample ID"] == sample_id
        ].copy()

        if ref_rows.empty:
            continue

        common_components = [
            component
            for component in scores
            if component in ref_rows["Component"].values
        ]

        if len(common_components) < 2:
            continue

        predicted_top = max(
            common_components,
            key=lambda component: scores[component]
        )

        ref_effects = {
            row["Component"]: float(row["Confidence Drop"])
            for _, row in ref_rows.iterrows()
        }

        reference_top = max(
            common_components,
            key=lambda component: ref_effects[component]
        )

        ranking_rows.append({
            "Method": method,
            "Sample ID": sample_id,
            "Predicted Top Component": predicted_top,
            "Occlusion Top Component": reference_top,
            "Top Component Agreement": int(
                predicted_top == reference_top
            ),
            "Occlusion Confidence Drop": ref_effects[predicted_top],
            "Reference Maximum Drop": ref_effects[reference_top]
        })

ranking_df = pd.DataFrame(ranking_rows)

summary = ranking_df.groupby("Method").agg(
    Samples=("Sample ID", "count"),
    Top_Component_Agreement=("Top Component Agreement", "mean"),
    Mean_Selected_Confidence_Drop=("Occlusion Confidence Drop", "mean"),
    Mean_Reference_Max_Drop=("Reference Maximum Drop", "mean")
)

print("=" * 90)
print("COMPONENT-LEVEL XAI FAITHFULNESS")
print("=" * 90)

print(summary.round(6).to_string())

output_path = RESULTS_DIR / "component_xai_faithfulness.csv"
ranking_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

COMPONENT-LEVEL XAI FAITHFULNESS
        Samples  Top_Component_Agreement  Mean_Selected_Confidence_Drop  Mean_Reference_Max_Drop
Method                                                                                          
IG           20                     0.95                       0.609841                 0.609848
LIME         20                     1.00                       0.609848                 0.609848
SHAP         20                     1.00                       0.609848                 0.609848

Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\component_xai_faithfulness.csv


In [45]:
rng = np.random.default_rng(SEED)

random_rows = []

for sample_id, group in reference_df.groupby("Sample ID"):

    available = group["Component"].tolist()

    if len(available) < 2:
        continue

    random_component = rng.choice(available)

    drops = dict(
        zip(group["Component"], group["Confidence Drop"])
    )

    reference_top = max(
        available,
        key=lambda component: drops[component]
    )

    random_rows.append({
        "Sample ID": int(sample_id),
        "Random Component": random_component,
        "Agreement": int(random_component == reference_top),
        "Confidence Drop": drops[random_component]
    })

random_df = pd.DataFrame(random_rows)

print("=" * 65)
print("RANDOM COMPONENT BASELINE")
print("=" * 65)

print("Samples:", len(random_df))
print("Top component agreement:", round(random_df["Agreement"].mean(), 4))
print("Mean confidence drop:", round(random_df["Confidence Drop"].mean(), 6))
print("=" * 65)

random_df.to_csv(
    RESULTS_DIR / "random_component_baseline.csv",
    index=False
)

RANDOM COMPONENT BASELINE
Samples: 20
Top component agreement: 0.55
Mean confidence drop: 0.424603


In [46]:
import string

def modify_url_component(url, component_name, seed=42):
    url = str(url)
    spans = get_component_spans(url)

    if component_name not in spans:
        return None

    start, end = spans[component_name]

    candidates = [
        i for i in range(start, end)
        if url[i].isalnum()
    ]

    if not candidates:
        return None

    rng = np.random.default_rng(seed)

    position = int(rng.choice(candidates))
    original_char = url[position]

    if original_char.isdigit():
        replacements = [
            c for c in string.digits
            if c != original_char
        ]
    else:
        replacements = [
            c for c in string.ascii_lowercase
            if c != original_char.lower()
        ]

    replacement = str(rng.choice(replacements))

    modified = (
        url[:position]
        + replacement
        + url[position + 1:]
    )

    return modified


sample_url = str(xai_df.iloc[0]["clean_url"])

print("Original:", sample_url)
print("Domain modification:",
      modify_url_component(sample_url, "domain"))

print("Path modification:",
      modify_url_component(sample_url, "path"))

Original: platteriverdecoyco.com/demo.html
Domain modification: puatteriverdecoyco.com/demo.html
Path modification: platteriverdecoyco.com/uemo.html


In [47]:
robustness_rows = []

for index, row in xai_df.iterrows():

    original_url = str(row["clean_url"])
    original_probs = predict_proba_lime([original_url])[0]

    original_class = int(np.argmax(original_probs))

    for component in ["domain", "path"]:

        modified_url = modify_url_component(
            original_url,
            component,
            seed=SEED + index
        )

        if modified_url is None:
            continue

        modified_probs = predict_proba_lime([modified_url])[0]

        modified_class = int(np.argmax(modified_probs))

        robustness_rows.append({
            "Sample ID": int(index),
            "Component Modified": component,
            "Original URL": original_url,
            "Modified URL": modified_url,
            "Original Prediction": original_class,
            "Modified Prediction": modified_class,
            "Prediction Flip": int(
                original_class != modified_class
            ),
            "Confidence Delta": float(
                original_probs[original_class]
                - modified_probs[original_class]
            )
        })

robustness_df = pd.DataFrame(robustness_rows)

robustness_summary = robustness_df.groupby(
    "Component Modified"
).agg(
    Samples=("Sample ID", "count"),
    Prediction_Flip_Rate=("Prediction Flip", "mean"),
    Mean_Confidence_Delta=("Confidence Delta", "mean")
)

print("=" * 80)
print("DISTILBERT PREDICTION ROBUSTNESS")
print("=" * 80)

print(robustness_summary.round(6).to_string())

output_path = RESULTS_DIR / "url_prediction_robustness.csv"

robustness_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

DISTILBERT PREDICTION ROBUSTNESS
                    Samples  Prediction_Flip_Rate  Mean_Confidence_Delta
Component Modified                                                      
domain                   20                   0.0               0.002333
path                     19                   0.0               0.001722

Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\url_prediction_robustness.csv


In [48]:
from sklearn.metrics.pairwise import cosine_similarity

ig_original_lookup = {
    int(item["sample_id"]): item
    for item in ig_component_results
}

stability_rows = []

for _, row in robustness_df.iterrows():

    if int(row["Prediction Flip"]) == 1:
        continue

    sample_id = int(row["Sample ID"])
    modified_url = str(row["Modified URL"])

    original_explanation = ig_original_lookup[sample_id]

    modified_explanation = explain_ig_components(
        modified_url,
        n_steps=64
    )

    original_scores = original_explanation["component_scores"]
    modified_scores = modified_explanation["component_scores"]

    common = sorted(
        set(original_scores) & set(modified_scores)
    )

    if not common:
        continue

    original_vector = np.array(
        [original_scores[c] for c in common]
    ).reshape(1, -1)

    modified_vector = np.array(
        [modified_scores[c] for c in common]
    ).reshape(1, -1)

    similarity = float(
        cosine_similarity(
            original_vector,
            modified_vector
        )[0, 0]
    )

    stability_rows.append({
        "Sample ID": sample_id,
        "Component Modified": row["Component Modified"],
        "Cosine Similarity": similarity,
        "Original URL": row["Original URL"],
        "Modified URL": modified_url
    })

stability_df = pd.DataFrame(stability_rows)

stability_summary = stability_df.groupby(
    "Component Modified"
).agg(
    Samples=("Sample ID", "count"),
    Mean_Cosine_Similarity=("Cosine Similarity", "mean"),
    Median_Cosine_Similarity=("Cosine Similarity", "median")
)

print("=" * 80)
print("IG EXPLANATION STABILITY")
print("=" * 80)

print(stability_summary.round(6).to_string())

output_path = RESULTS_DIR / "ig_explanation_stability.csv"

stability_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

IG EXPLANATION STABILITY
                    Samples  Mean_Cosine_Similarity  Median_Cosine_Similarity
Component Modified                                                           
domain                   20                0.950391                  0.997336
path                     19                0.993002                  0.999400

Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\ig_explanation_stability.csv


In [49]:
from sklearn.metrics.pairwise import cosine_similarity

stability_comparison_rows = []

for method, explanations in [
    ("LIME", lime_component_results),
    ("SHAP", shap_component_results)
]:

    original_lookup = {
        int(item["sample_id"]): item
        for item in explanations
    }

    for _, row in robustness_df.iterrows():

        if int(row["Prediction Flip"]) == 1:
            continue

        sample_id = int(row["Sample ID"])
        modified_url = str(row["Modified URL"])

        original = original_lookup[sample_id]

        if method == "LIME":
            modified = explain_lime_components(
                modified_url,
                num_samples=256,
                seed=SEED + sample_id
            )
        else:
            modified = explain_shap_components(modified_url)

        original_scores = original["component_scores"]
        modified_scores = modified["component_scores"]

        common = sorted(
            set(original_scores) & set(modified_scores)
        )

        if not common:
            continue

        original_vector = np.array(
            [original_scores[c] for c in common],
            dtype=float
        )

        modified_vector = np.array(
            [modified_scores[c] for c in common],
            dtype=float
        )

        original_norm = np.linalg.norm(original_vector)
        modified_norm = np.linalg.norm(modified_vector)

        if original_norm < 1e-8 or modified_norm < 1e-8:
            similarity = np.nan
        else:
            similarity = float(
                np.dot(original_vector, modified_vector)
                / (original_norm * modified_norm)
            )

        stability_comparison_rows.append({
            "Method": method,
            "Sample ID": sample_id,
            "Component Modified": row["Component Modified"],
            "Cosine Similarity": similarity,
            "Original URL": row["Original URL"],
            "Modified URL": modified_url
        })

comparison_df = pd.DataFrame(stability_comparison_rows)

# Include previously calculated IG stability results.
ig_stability_copy = stability_df.copy()
ig_stability_copy["Method"] = "IG"

all_stability_df = pd.concat(
    [ig_stability_copy, comparison_df],
    ignore_index=True
)

print("=" * 90)
print("XAI EXPLANATION STABILITY COMPARISON")
print("=" * 90)

summary = all_stability_df.groupby(
    ["Method", "Component Modified"]
).agg(
    Samples=("Sample ID", "count"),
    Valid_Similarities=("Cosine Similarity", "count"),
    Mean_Cosine=("Cosine Similarity", "mean"),
    Median_Cosine=("Cosine Similarity", "median")
)

print(summary.round(6).to_string())

output_path = RESULTS_DIR / "xai_component_stability_comparison.csv"

all_stability_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

XAI EXPLANATION STABILITY COMPARISON
                           Samples  Valid_Similarities  Mean_Cosine  Median_Cosine
Method Component Modified                                                         
IG     domain                   20                  20     0.950391       0.997336
       path                     19                  19     0.993002       0.999400
LIME   domain                   20                  20     0.995628       1.000000
       path                     19                  19     0.988271       0.999914
SHAP   domain                   20                  20     0.998118       1.000000
       path                     19                  19     0.987618       0.999943

Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\xai_component_stability_comparison.csv


In [50]:
top_stability_rows = []

for method, explanations in [
    ("IG", ig_component_results),
    ("LIME", lime_component_results),
    ("SHAP", shap_component_results)
]:

    original_lookup = {
        int(item["sample_id"]): item
        for item in explanations
    }

    for _, row in robustness_df.iterrows():

        if int(row["Prediction Flip"]) == 1:
            continue

        sample_id = int(row["Sample ID"])
        modified_url = str(row["Modified URL"])

        original = original_lookup[sample_id]

        if method == "IG":
            modified = explain_ig_components(modified_url, n_steps=64)
        elif method == "LIME":
            modified = explain_lime_components(
                modified_url,
                num_samples=256,
                seed=SEED + sample_id
            )
        else:
            modified = explain_shap_components(modified_url)

        original_scores = original["component_scores"]
        modified_scores = modified["component_scores"]

        common = sorted(
            set(original_scores) & set(modified_scores)
        )

        if len(common) < 2:
            continue

        original_top = max(
            common,
            key=lambda c: original_scores[c]
        )

        modified_top = max(
            common,
            key=lambda c: modified_scores[c]
        )

        top_stability_rows.append({
            "Method": method,
            "Sample ID": sample_id,
            "Component Modified": row["Component Modified"],
            "Original Top": original_top,
            "Modified Top": modified_top,
            "Top Component Stable": int(
                original_top == modified_top
            )
        })

top_stability_df = pd.DataFrame(top_stability_rows)

top_summary = top_stability_df.groupby(
    ["Method", "Component Modified"]
).agg(
    Samples=("Sample ID", "count"),
    Top_Component_Stability=("Top Component Stable", "mean")
)

print("=" * 85)
print("TOP COMPONENT STABILITY")
print("=" * 85)

print(top_summary.round(6).to_string())

output_path = RESULTS_DIR / "xai_top_component_stability.csv"

top_stability_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

TOP COMPONENT STABILITY
                           Samples  Top_Component_Stability
Method Component Modified                                  
IG     domain                   20                     0.95
       path                     19                     1.00
LIME   domain                   20                     1.00
       path                     19                     1.00
SHAP   domain                   20                     1.00
       path                     19                     1.00

Saved: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\xai\xai_top_component_stability.csv


In [51]:
import pandas as pd
import numpy as np

runtime = pd.read_csv(
    RESULTS_DIR / "xai_runtime_comparison.csv"
)

faithfulness = pd.read_csv(
    RESULTS_DIR / "component_xai_faithfulness.csv"
)

stability = pd.read_csv(
    RESULTS_DIR / "xai_component_stability_comparison.csv"
)

top_stability = pd.read_csv(
    RESULTS_DIR / "xai_top_component_stability.csv"
)

print("=" * 85)
print("FINAL XAI RESEARCH SUMMARY")
print("=" * 85)

for method in ["IG", "LIME", "SHAP"]:

    faith = faithfulness[
        faithfulness["Method"] == method
    ]

    stab = stability[
        stability["Method"] == method
    ]

    top = top_stability[
        top_stability["Method"] == method
    ]

    print(f"\n{method}")
    print("-" * 40)

    print(
        "Occlusion ranking agreement:",
        round(faith["Top Component Agreement"].mean(), 4)
    )

    print(
        "Mean explanation cosine similarity:",
        round(stab["Cosine Similarity"].mean(), 4)
    )

    print(
        "Top component stability:",
        round(top["Top Component Stable"].mean(), 4)
    )

print("\nResearch summary completed.")

FINAL XAI RESEARCH SUMMARY

IG
----------------------------------------
Occlusion ranking agreement: 0.95
Mean explanation cosine similarity: 0.9712
Top component stability: 0.9744

LIME
----------------------------------------
Occlusion ranking agreement: 1.0
Mean explanation cosine similarity: 0.992
Top component stability: 1.0

SHAP
----------------------------------------
Occlusion ranking agreement: 1.0
Mean explanation cosine similarity: 0.993
Top component stability: 1.0

Research summary completed.
